## Software Design Patterns in Python

### Day 2: Creational Patterns, Accessors & AOP
#### Module 2: The Python Data Model & Protocols
 - **Descriptor Protocol:** Implementing `__get__`, `__set__`, and `__getattr__` / `__setattr__` / `__delattr__` (The foundation of Python properties).
 - **Property accessors:** Using the builtin `@property` accessors.
 - **Context Management:** Creating custom context managers with `__enter__` and `__exit__`.
 - **Callable Objects:** Using `__call__` to make instances behave like functions.
 - **Review:** How these hooks replace "boilerplate" code found in Java/C++ patterns.

#### Module 3: Decorators & AOP (Aspect Oriented Programming)
 - **Decorator Deep Dive:** Function vs. Class decorators.
 - **Metadata Preservation:** Using `functools.wraps` to preserve introspection.
 - **Pattern Application:** Logging, Caching (Memoization), and Access Control (Proxying).
 - **Parameterized Decorators:** Building decorators that accept configuration arguments.

#### Module 4: Pythonic Creational Patterns
 - **Singleton:** Four implementation styles (Module-level, Decorator, Metaclass, Monostate).
 - **Factory Method & Abstract Factory:** Leveraging dynamic typing and classes-as-objects.
 - **Builder:** Implementing fluent interfaces and method chaining.
 - **Prototype:** Effective use of `copy.deepcopy` vs custom cloning methods.



---

Solution to the Run() process exercise

In [ ]:
import subprocess
import sys

class Run:
    def __init__(self, cmd):
        self.cmd = cmd
    
    def __or__(self, other):
        return Run(f"{self.cmd} | {other.cmd}")
    
    def __gt__(self, out):
        subprocess.run(self.cmd, shell=True, stdout=out, stderr=out)

    def run(self):
        self > sys.stdout

if __name__ == "__main__":
    Run("ls") | Run("wc -l") > sys.stdout
    Run("ls -lrt") > sys.stdout
    Run("ls -lrt").run()

#### Another exercise: Implement a Queue with intuitive operator support to add / remove elements.

In [ ]:
from collections import deque

class Queue:
    def __init__(self, *args, **kwargs):
        self.queue = deque(*args, **kwargs)

    # TODO: Implement the '>>' operator to enqueue an element to the queue
    
    # TODO: Implement the '<<' operator to dequeue an element from the queue

    def __str__(self):
        return f"Queue({list(self.queue)})"

if __name__ == '__main__':
    q = Queue()
    2 >> q  # Enqueue element [2]
    3 >> q  # [3, 2]
    4 >> q  # [4, 3, 2]

    q << 10 # Enqueue element 10 [4, 3, 2, 10]
    q << 11 # [4, 3, 2, 10, 11]
    q << 12 # [4, 3, 2, 10, 11, 12]

    r = Queue()
    q >> r # Dequeue element from q and enqueue to r [12], q=[4, 3, 2, 10, 11]
    q >> r # [11, 12], q=[4, 3, 2, 10]
    q >> r # [10, 11, 12], q=[4, 3, 2]

    print(q)
    print(r)


---

#### Type conversion / coercion of Python objects

In [ ]:
class User:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"<User: name='{self.name}'>"

    def __repr__(self):
        return f"User(name='{self.name}')"  # Official representation (should mimic python expression)

u1 = User("Alice")
u2 = User("Bob")
print(u1) # str(u1)
print(u2)
u1  # repr(u1)

<User: name='Alice'>
<User: name='Bob'>


User(name='Alice')

In [ ]:
# Real-world example of __str__ and __repr__:
import requests

res = requests.get("https://python.org/")
res

<Response [200]>

In [ ]:
def foo(): pass

foo

<function __main__.foo()>

#### Implementing support for int() and float() conversions

In [8]:
class Stats:
    def __init__(self, temperature, humidity):
        self.temperature = temperature
        self.humidity = humidity

    def __int__(self):
        return int(self.humidity)
    
    def __float__(self):
        return float(self.temperature)
    
    def __add__(self, other):
        if isinstance(other, Stats):
            return Stats(self.temperature + other.temperature, self.humidity + other.humidity)
        elif isinstance(other, int) :
            return Stats(self.temperature, self.humidity + other)
        elif isinstance(other, float):
            return Stats(self.temperature + other, self.humidity)
        else:
            return NotImplemented

    __radd__ = __add__  # For commutative addition    

    def __str__(self):
        return f"Stats<temperature: {self.temperature}, humidity: {self.humidity}>"
    
    def __repr__(self):
        return f"Stats(temperature={self.temperature}, humidity={self.humidity})"

s = Stats(22.5, 60)
print(s)
print(int(s))
print(float(s))
print(10 + s)
print(5.5 + s)
# NOTE: int() and float() output can seem ambiguous, use with caution

Stats<temperature: 22.5, humidity: 60>
60
22.5
Stats<temperature: 22.5, humidity: 70>
Stats<temperature: 28.0, humidity: 60>


In [ ]:
from collections import deque

class Queue:
    def __init__(self, *args, **kwargs):
        self.queue = deque(*args, **kwargs)

    def __rrshift__(self, v): # 2 >> q
        self.queue.appendleft(v)

    def __lshift__(self, v): # q << 10
        self.queue.append(v)

    def __rshift__(self, dest): # q >> r
        if type(dest) is type(self):
            dest.queue.appendleft(self.queue.pop())
        elif hasattr(dest, 'appendleft'):
            dest.appendleft(self.queue.pop())
        elif hasattr(dest, 'insert'):
            dest.insert(0, self.queue.pop())
        elif dest is None:
            self.queue.pop()
        else:
            raise TypeError("Destination must be a Queue, deque, list, or None")
        return dest

    def __repr__(self):
        return f"Queue({list(self.queue)})"

In [ ]:
q1 = Queue([10, 20, 30, 40])
print(q1)

Queue([10, 20, 30, 40])


----